In [0]:
# Widget for selecting catalog (default: main)
dbutils.widgets.text("catalog_use", "main", "Catalog")
# Widget for selecting schema (default: dev_matthew_giglia_ames_housing)
dbutils.widgets.text("schema_use", "dev_matthew_giglia_ames_housing", "Schema")
# Widget for ML model name (default: dev_matthew_giglia_house_price_prediction)
dbutils.widgets.text("ml_model_name_use", "dev_matthew_giglia_house_price_prediction", "ML Model Name")
# Widget for ML model storage location (default: /Volumes/main/dev_matthew_giglia_ames_housing/ml_artifacts)
dbutils.widgets.text("ml_model_storage_location", "/Volumes/main/dev_matthew_giglia_ames_housing/ml_artifacts", "ML Model Storage Location")
# Widget for ML experiment name (default: /Workspace/experiments/[dev matthew_giglia] house_price_prediction)
dbutils.widgets.text("ml_model_experiments_name_use", "/Workspace/experiments/[dev matthew_giglia] house_price_prediction", "ML Model Experiments Name")

In [0]:
# Retrieve the current values from the widgets defined above
catalog_use = dbutils.widgets.get("catalog_use")
schema_use = dbutils.widgets.get("schema_use")
ml_model_name_use = dbutils.widgets.get("ml_model_name_use")
ml_model_storage_location = dbutils.widgets.get("ml_model_storage_location")
ml_model_experiments_name_use = dbutils.widgets.get("ml_model_experiments_name_use")

In [0]:
# Print the current configuration values for verification
# This printout displays the current settings for the ML pipeline, including the catalog, schema, model name, storage location, and experiment names.
print(f"""
	catalog_use: {catalog_use}
	schema_use: {schema_use}
	ml_model_name_use: {ml_model_name_use}
	ml_model_storage_location: {ml_model_storage_location}
	ml_model_experiments_name_use: {ml_model_experiments_name_use}
""")

In [0]:
# Load the sale_prices table from the catalog
table_name = f"{catalog_use}.{schema_use}.sale_prices"
df = spark.table(table_name)

# Display basic information
row_count = df.count()
column_count = len(df.columns)

print(f"Dataset: {table_name}")
print(f"Total rows: {row_count:,}")
print(f"Total columns: {column_count}")
print(f"\nSchema:")
df.printSchema()

In [0]:
# Display first few rows to understand the data structure
display(df.limit(10))

In [0]:
from pyspark.sql import functions as F

# Calculate comprehensive statistics for sale_price
sale_price_stats = df.select(
	F.count('sale_price').alias('count'),
	F.countDistinct('sale_price').alias('unique_values'),
	F.sum(F.when(F.col('sale_price').isNull(), 1).otherwise(0)).alias('null_count'),
	F.min('sale_price').alias('min'),
	F.expr('percentile(sale_price, 0.25)').alias('q1'),
	F.expr('percentile(sale_price, 0.50)').alias('median'),
	F.mean('sale_price').alias('mean'),
	F.expr('percentile(sale_price, 0.75)').alias('q3'),
	F.max('sale_price').alias('max'),
	F.stddev('sale_price').alias('std_dev')
).collect()[0]

print("Sale Price Statistics:")
print(f"Count: {sale_price_stats['count']:,}")
print(f"Unique values: {sale_price_stats['unique_values']:,}")
print(f"Null count: {sale_price_stats['null_count']:,}")
print(f"\nDistribution:")
print(f"Min: ${sale_price_stats['min']:,.0f}")
print(f"Q1 (25%): ${sale_price_stats['q1']:,.0f}")
print(f"Median (50%): ${sale_price_stats['median']:,.0f}")
print(f"Mean: ${sale_price_stats['mean']:,.2f}")
print(f"Q3 (75%): ${sale_price_stats['q3']:,.0f}")
print(f"Max: ${sale_price_stats['max']:,.0f}")
print(f"Std Dev: ${sale_price_stats['std_dev']:,.2f}")

# Calculate IQR for outlier detection
iqr = sale_price_stats['q3'] - sale_price_stats['q1']
lower_bound = sale_price_stats['q1'] - 1.5 * iqr
upper_bound = sale_price_stats['q3'] + 1.5 * iqr
print(f"\nOutlier bounds (IQR method):")
print(f"Lower bound: ${lower_bound:,.0f}")
print(f"Upper bound: ${upper_bound:,.0f}")

In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Convert to pandas for plotting
sale_price_pd = df.select('sale_price').toPandas()

# Create subplots
fig = make_subplots(
	rows=1, cols=2,
	subplot_titles=('Sale Price Distribution', 'Sale Price Box Plot')
)

# Histogram
fig.add_trace(
	go.Histogram(x=sale_price_pd['sale_price'], nbinsx=50, name='Sale Price'),
	row=1, col=1
)

# Box plot
fig.add_trace(
	go.Box(y=sale_price_pd['sale_price'], name='Sale Price'),
	row=1, col=2
)

fig.update_layout(
	height=400,
	showlegend=False,
	template='plotly_white'
)

fig.update_xaxes(title_text="Sale Price ($)", row=1, col=1)
fig.update_yaxes(title_text="Frequency", row=1, col=1)
fig.update_yaxes(title_text="Sale Price ($)", row=1, col=2)

display(fig)

In [0]:
# Count outliers using IQR method
outliers_df = df.filter(
	(F.col('sale_price') < lower_bound) | (F.col('sale_price') > upper_bound)
)

outlier_count = outliers_df.count()
outlier_percentage = (outlier_count / sale_price_stats['count']) * 100

print(f"Outliers detected: {outlier_count} ({outlier_percentage:.2f}%)")

if outlier_count > 0:
	print(f"\nTop 10 highest sale prices (potential outliers):")
	display(df.select('property_id', 'sale_price', 'neighborhood', 'overall_quality', 'above_ground_living_area')
			 .orderBy(F.col('sale_price').desc())
			 .limit(10))

In [0]:
# Calculate missing value counts and percentages for all columns
# Exclude property_id (primary key) and _rescued_data (non-feature column)
feature_cols = [col for col in df.columns if col not in ['property_id', '_rescued_data']]

missing_data = []
for col in feature_cols:
	null_count = df.filter(F.col(col).isNull()).count()
	null_percentage = (null_count / row_count) * 100
	missing_data.append({
		'feature': col,
		'null_count': null_count,
		'null_percentage': null_percentage
	})

# Convert to DataFrame and sort by null percentage
missing_df = spark.createDataFrame(missing_data).orderBy(F.col('null_percentage').desc())

print(f"Features with missing values (out of {len(feature_cols)} total features):")
features_with_nulls = missing_df.filter(F.col('null_count') > 0).count()
print(f"{features_with_nulls} features have missing values\n")

# Display top features with missing values
print("Top 20 features with highest missing value percentages:")
display(missing_df.filter(F.col('null_count') > 0).limit(20))

In [0]:
import plotly.express as px

# Get features with missing values for visualization
missing_pd = missing_df.filter(F.col('null_count') > 0).toPandas()

if len(missing_pd) > 0:
	# Create bar chart of missing values
	fig = px.bar(
		missing_pd.head(30),
		x='null_percentage',
		y='feature',
		orientation='h',
		title='Missing Value Percentages by Feature (Top 30)',
		labels={'null_percentage': 'Missing %', 'feature': 'Feature'},
		template='plotly_white'
	)
	fig.update_layout(height=800, yaxis={'categoryorder': 'total ascending'})
	display(fig)
else:
	print("No missing values found in the dataset!")

In [0]:
# Categorize features by missing value percentage
high_missing = missing_df.filter(F.col('null_percentage') > 50).count()
moderate_missing = missing_df.filter((F.col('null_percentage') > 10) & (F.col('null_percentage') <= 50)).count()
low_missing = missing_df.filter((F.col('null_percentage') > 0) & (F.col('null_percentage') <= 10)).count()
no_missing = missing_df.filter(F.col('null_percentage') == 0).count()

print("Missing Value Summary by Severity:")
print(f"\nHigh missing (>50%): {high_missing} features")
print(f"Moderate missing (10-50%): {moderate_missing} features")
print(f"Low missing (<10%): {low_missing} features")
print(f"No missing values: {no_missing} features")
print(f"\nTotal features analyzed: {len(feature_cols)}")

# Show features with high missing values
if high_missing > 0:
	print("\nFeatures with >50% missing values (consider dropping):")
	display(missing_df.filter(F.col('null_percentage') > 50))

In [0]:
# Identify numerical columns (excluding property_id and _rescued_data)
numerical_cols = []
for field in df.schema.fields:
	if field.name not in ['property_id', '_rescued_data', 'sale_price']:
		if field.dataType.typeName() in ['integer', 'long', 'double', 'float', 'decimal']:
			numerical_cols.append(field.name)

print(f"Found {len(numerical_cols)} numerical features (excluding target variable)")
print(f"\nNumerical features: {numerical_cols[:10]}...")

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import numpy as np

# Prepare data for correlation calculation (drop nulls for correlation)
df_for_corr = df.select(['sale_price'] + numerical_cols).na.drop()

# Calculate Pearson correlation for each numerical feature with sale_price
correlations = []
for col in numerical_cols:
	corr_value = df_for_corr.stat.corr('sale_price', col)
	correlations.append({
		'feature': col,
		'correlation': corr_value,
		'abs_correlation': abs(corr_value)
	})

# Convert to DataFrame and sort by absolute correlation
corr_df = spark.createDataFrame(correlations).orderBy(F.col('abs_correlation').desc())

print("Top 20 features most correlated with sale_price:")
display(corr_df.limit(20))

In [0]:
# Get top correlations for visualization
top_corr_pd = corr_df.limit(15).toPandas()

# Create horizontal bar chart
fig = px.bar(
	top_corr_pd,
	x='correlation',
	y='feature',
	orientation='h',
	title='Top 15 Features by Correlation with Sale Price',
	labels={'correlation': 'Pearson Correlation', 'feature': 'Feature'},
	template='plotly_white',
	color='correlation',
	color_continuous_scale='RdBu_r',
	color_continuous_midpoint=0
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
display(fig)

In [0]:
# Get top 6 most correlated features for detailed analysis
top_features = [row['feature'] for row in corr_df.limit(6).collect()]

print(f"Analyzing distributions of top 6 correlated features:")
for feature in top_features:
	print(f"  - {feature}")

# Create distribution plots for top features
from plotly.subplots import make_subplots

fig = make_subplots(
	rows=2, cols=3,
	subplot_titles=top_features
)

for idx, feature in enumerate(top_features):
	row = (idx // 3) + 1
	col = (idx % 3) + 1
	
	feature_data = df.select(feature).na.drop().toPandas()
	
	fig.add_trace(
		go.Histogram(x=feature_data[feature], name=feature, showlegend=False),
		row=row, col=col
	)

fig.update_layout(
	height=600,
	title_text="Distribution of Top 6 Correlated Features",
	template='plotly_white'
)

display(fig)

In [0]:
# Identify categorical (string) columns
categorical_cols = []
for field in df.schema.fields:
	if field.name not in ['property_id', '_rescued_data', 'sale_price']:
		if field.dataType.typeName() == 'string':
			categorical_cols.append(field.name)

print(f"Found {len(categorical_cols)} categorical features")
print(f"\nCategorical features: {categorical_cols[:15]}...")

In [0]:
# Calculate cardinality (number of unique values) for each categorical feature
cardinality_data = []
for col in categorical_cols:
	unique_count = df.select(col).distinct().count()
	null_count = df.filter(F.col(col).isNull()).count()
	cardinality_data.append({
		'feature': col,
		'unique_values': unique_count,
		'null_count': null_count
	})

# Convert to DataFrame and sort by unique values
cardinality_df = spark.createDataFrame(cardinality_data).orderBy(F.col('unique_values').desc())

print("Categorical features by cardinality:")
display(cardinality_df)

# Categorize by cardinality
high_cardinality = cardinality_df.filter(F.col('unique_values') > 10).count()
low_cardinality = cardinality_df.filter(F.col('unique_values') <= 10).count()

print(f"\nHigh cardinality (>10 unique values): {high_cardinality} features")
print(f"Low cardinality (≤10 unique values): {low_cardinality} features")

In [0]:
# Select key categorical features for analysis (low to medium cardinality)
key_categorical = ['neighborhood', 'overall_quality', 'exterior_quality', 'kitchen_quality', 
				  'basement_quality', 'building_type', 'house_style', 'ms_zoning']

# Filter to only those that exist in our categorical columns
key_categorical = [col for col in key_categorical if col in categorical_cols or col in numerical_cols]

print(f"Analyzing {len(key_categorical)} key categorical features:")
for feature in key_categorical:
	print(f"  - {feature}")

In [0]:
# Analyze neighborhood impact on sale price
neighborhood_stats = df.groupBy('neighborhood').agg(
	F.count('sale_price').alias('count'),
	F.mean('sale_price').alias('avg_sale_price'),
	F.stddev('sale_price').alias('std_sale_price')
).orderBy(F.col('avg_sale_price').desc())

print("Average sale price by neighborhood (top 15):")
display(neighborhood_stats.limit(15))

In [0]:
# Visualize neighborhood impact
neighborhood_pd = neighborhood_stats.toPandas()

fig = px.bar(
	neighborhood_pd.head(20),
	x='avg_sale_price',
	y='neighborhood',
	orientation='h',
	title='Average Sale Price by Neighborhood (Top 20)',
	labels={'avg_sale_price': 'Average Sale Price ($)', 'neighborhood': 'Neighborhood'},
	template='plotly_white',
	color='avg_sale_price',
	color_continuous_scale='Viridis'
)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
display(fig)

In [0]:
# Analyze quality-related categorical features
quality_features = ['exterior_quality', 'kitchen_quality', 'basement_quality']

for feature in quality_features:
	if feature in categorical_cols:
		quality_stats = df.groupBy(feature).agg(
			F.count('sale_price').alias('count'),
			F.mean('sale_price').alias('avg_sale_price')
		).orderBy(F.col('avg_sale_price').desc())
		
		print(f"\nAverage sale price by {feature}:")
		display(quality_stats)

## Exploratory Data Analysis Summary

### Dataset Overview
* **Total Records**: 2,930 properties
* **Total Features**: 80 (36 numerical, 43 categorical, 1 target)
* **Target Variable**: `sale_price` (mean: $180,796, median: $160,000)
* **Outliers**: 137 properties (4.68%) above $339,500

### Key Findings

#### 1. Target Variable (`sale_price`)
* Right-skewed distribution with some high-value outliers
* Range: $12,789 to $755,000
* Consider log transformation for modeling to handle skewness

#### 2. Missing Values
* **21 features** have missing values
* **Moderate missing** (10-50%): `lot_frontage` (16.7%)
* **Low missing** (<10%): 20 features, mostly garage and basement-related
* **No high missing** (>50%) features - good data quality overall

#### 3. Top Predictive Numerical Features (by correlation)
1. `overall_quality` (0.804) - Strongest predictor
2. `above_ground_living_area` (0.714)
3. `garage_car_capacity` (0.661)
4. `garage_area` (0.648)
5. `total_basement_sf` (0.646)
6. `first_floor_sf` (0.639)

#### 4. Important Categorical Features
* **Neighborhood**: Significant price variation (NoRidge avg: $330K vs MeadowV: ~$90K)
* **Quality features**: `exterior_quality`, `kitchen_quality`, `basement_quality` show strong relationships
* Most categorical features have low cardinality (≤10 unique values) - suitable for one-hot encoding

### Feature Engineering Recommendations

#### A. Target Transformation
* Apply **log transformation** to `sale_price` to normalize distribution
* This will help linear models and improve prediction accuracy

#### B. Missing Value Handling
* **Numerical features**: Impute with median or mean
* **Categorical features**: Create 'Missing' category or use mode
* **`lot_frontage`**: Consider neighborhood-based imputation (16.7% missing)

#### C. Feature Creation
1. **Total Square Footage**: `total_basement_sf` + `above_ground_living_area`
2. **Total Bathrooms**: `full_bathrooms` + 0.5 * `half_bathrooms` + `basement_full_bathrooms` + 0.5 * `basement_half_bathrooms`
3. **House Age**: Current year - `year_built`
4. **Years Since Remodel**: Current year - `year_remodeled`
5. **Total Porch Area**: Sum of all porch square footages
6. **Has Pool**: Binary indicator from `pool_area`
7. **Has Garage**: Binary indicator from `garage_area`
8. **Quality Score**: Composite of quality-related features

#### D. Encoding Strategy
* **One-hot encoding** for low-cardinality categorical features (≤10 unique values)
* **Target encoding** or **frequency encoding** for high-cardinality features (e.g., `neighborhood`)
* **Ordinal encoding** for quality features (Ex > Gd > TA > Fa > Po)

#### E. Feature Selection
* Remove `property_id` (identifier) and `_rescued_data` (non-feature)
* Consider removing highly correlated features to reduce multicollinearity
* Focus on top 20-30 features based on correlation and domain knowledge

#### F. Outlier Treatment
* Consider capping extreme values at 99th percentile
* Or use robust models (tree-based) that handle outliers naturally

### Next Steps for Modeling
1. Implement feature engineering pipeline
2. Split data: 60% train, 20% validation, 20% test
3. Scale numerical features using StandardScaler
4. Try multiple algorithms: Linear Regression, Random Forest, XGBoost, LightGBM
5. Use hyperparameter tuning with Hyperopt
6. Track experiments with MLflow